# LaTeX tables

Builds the paper's LaTeX tables. Two parts:
1. **Cue-words summary** — per-sample cue words grouped by semantic category (word lists, counts),
   exported as CSV and LaTeX. Requires network access for `deep_translator`.
2. **Network-feature tables** — the per-keyword and total network-feature tables, with cue-word
   hubs italicised.

In [1]:
import ast
import re
import numpy as np
import pandas as pd
from deep_translator import GoogleTranslator

from config import RESSERENO_ROOT, MATH_ANXIETY_ROOT

pd.set_option("display.max_colwidth", None)

FORMATTING = RESSERENO_ROOT / "out" / "formatting"
MA_FORMATTING = MATH_ANXIETY_ROOT / "out" / "formatting"
path_rassereno_analysis = RESSERENO_ROOT / "out" / "analysis"

## Load the association files

In [2]:
assoc_2023 = pd.read_csv(
    MA_FORMATTING / "humans_2023" / "associations_valence_2023_humans.csv"
)
assoc_2024 = pd.read_csv(
    MA_FORMATTING / "humans_2024" / "associations_valence_2024_humans.csv"
)
assoc_BA_2024 = pd.read_csv(
    FORMATTING / "psychology_BA_2024" / "associations_valence_2024.csv"
)

assoc_2025 = pd.read_csv(
    FORMATTING / "all_students_2025" / "associations_valence_task_2025.csv"
)
assoc_psychology_2025 = assoc_2025[
    assoc_2025["participant_id"].str.startswith("interfacce")
]

## Semantic categories

In [3]:
# Italian cue words grouped by semantic category (2023-2025 samples)
CATEGORY_LISTS = {
    "STEM Disciplines": [
        "algoritmo",
        "analisi",
        "biologia",
        "calcolo",
        "dimostrazione",
        "equazione",
        "fisica",
        "informatica",
        "ingegneria",
        "matematica",
        "matematico/a",
        "numeri",
        "performance_in_matematica",
        "statistica",
        "tecnologia",
        "test_di_matematica",
        "utilità_della_matematica",
        "dati",
        "grafico",
        "ipotesi",
        "laboratorio",
        "logica",
        "ricerca",
        "scienza",
        "scienziata",
        "scienziato",
        "scoperta",
        "esperimento",
        "innovazione",
        "neuroscienze",
    ],
    "Academic Environment": [
        "classe",
        "compagni",
        "docente",
        "genitore",
        "genitori",
        "insegnante",
        "studente",
        "studente/ssa",
        "professore",
        "lavagna",
        "mappa_concettuale",
        "note",
        "scuola",
        "università",
        "laurea",
    ],
    "Evaluation": [
        "esame",
        "esercitazione",
        "prova",
        "test",
        "media",
        "valutazione",
        "voto",
    ],
    "Learning and Motivation": [
        "cambio_di_piani",
        "compito",
        "concentrazione",
        "confusione",
        "conoscenza",
        "creatività",
        "curiosità",
        "distrazione",
        "divertimento",
        "errori",
        "fissare_un_obiettivo",
        "impegno",
        "importanza",
        "intelligenza",
        "memorizzare",
        "motivazione",
        "pianificare",
        "problema",
        "programma",
        "programmare",
        "programmazione",
        "revisionare",
        "rimandare",
        "scadenza",
        "sfida",
        "sforzo",
        "soluzione",
        "studiare",
        "successo",
        "tenere_traccia",
        "futuro",
        "modello",
        "sapere",
        "regole",
        "relazione",
        "sviluppo",
        "passione",
        "lavoro",
    ],
    "Mental Health": [
        "ansia",
        "arte",
        "attaccamento",
        "benessere",
        "cognizione",
        "counselling",
        "emozione",
        "fallimento",
        "frustrazione",
        "imbarazzo",
        "meditazione",
        "mente",
        "noia",
        "panico",
        "paura",
        "personalità",
        "psicologia",
        "psicopatologia",
        "psicoterapia",
        "terapeuta",
        "terapia",
        "stress",
        "comportamento",
        "vita",
    ],
}

# 2019 experts sample: already-English cue words per category (no translation needed)
CATEGORY_LISTS_2019 = {
    "STEM Disciplines": [
        "science",
        "physics",
        "mathematics",
        "chemistry",
        "biology",
        "statistics",
        "electricity",
        "computer",
        "evolution",
        "dna",
        "experiment",
        "cell",
        "derivative",
        "space",
        "graph",
        "theorem",
        "trigonometry",
        "integral",
        "model",
        "complex",
        "chaos",
        "solid",
        "number",
        "hub",
        "system",
        "network",
        "theory",
    ],
    "Academic Environment": [
        "university",
        "school",
        "conference",
        "lab",
        "studies",
        "teaching",
        "student",
        "degree",
        "mathematician",
        "prisoner",
    ],
    "Evaluation": ["analysis", "game", "word", "loop", "market"],
    "Learning and Motivation": ["language", "person", "art"],
    "Mental Health": ["life", "brain", "water", "scientific", "cloud"],
}
CATEGORY_LISTS_2019 = {
    k: [w.capitalize() for w in v] for k, v in CATEGORY_LISTS_2019.items()
}

## Build the per-sample category table

In [4]:
def cue_words_summary(df):
    """One-row dataframe: for each category, the (English) cue words present in `df`."""
    present = set(df["cue_word"])
    grouped = {
        cat: [w for w in words if w in present] for cat, words in CATEGORY_LISTS.items()
    }
    translated = {
        cat: [
            GoogleTranslator(source="it", target="en").translate(w.capitalize())
            for w in words
        ]
        for cat, words in grouped.items()
    }
    n_words = sum(len(v) for v in grouped.values())
    print(f"{n_words} cue words for {df['participant_id'].nunique()} participants")
    return pd.DataFrame([translated])


rows = [
    pd.DataFrame([CATEGORY_LISTS_2019]),  # 2019 experts (already English)
    cue_words_summary(assoc_2023),
    cue_words_summary(assoc_2024),
    cue_words_summary(assoc_BA_2024),
    cue_words_summary(assoc_psychology_2025),
]
df_summary = pd.concat(rows, axis=0, ignore_index=True)

51 cue words for 57 participants
40 cue words for 70 participants
41 cue words for 166 participants
42 cue words for 23 participants


## Format, count and export

In [8]:
def count_words(cell):
    if isinstance(cell, (list, np.ndarray)):
        return len(cell)
    if pd.isna(cell) or str(cell).strip() == "":
        return 0
    return len([w for w in str(cell).split(",") if w.strip()])


# join the per-category word lists into comma-separated strings
for col in df_summary.columns:
    df_summary[col] = df_summary[col].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else x
    )

df_summary.insert(0, "Set", df_summary.index + 1)
df_summary["Total"] = df_summary[[c for c in df_summary.columns if c != "Set"]].apply(
    lambda row: sum(count_words(cell) for cell in row), axis=1
)

# df_summary.to_csv(FORMATTING / "cue_words_summary.csv", index=False)
df_summary

,Set,STEM Disciplines,Academic Environment,Evaluation,Learning and Motivation,Mental Health,Total
0,1,"Science, Physics, Mathematics, Chemistry, Biology, Statistics, Electricity, Computer, Evolution, Dna, Experiment, Cell, Derivative, Space, Graph, Theorem, Trigonometry, Integral, Model, Complex, Chaos, Solid, Number, Hub, System, Network, Theory","University, School, Conference, Lab, Studies, Teaching, Student, Degree, Mathematician, Prisoner","Analysis, Game, Word, Loop, Market","Language, Person, Art","Life, Brain, Water, Scientific, Cloud",51
1,2,"Algorithm, Biology, Calculation, Demonstration, Equation, Physics, Engineering, Mathematics, Mathematician, Numbers, Performance_in_mathematics, Statistics, Technology, Math_test, Utility_of_mathematics","Class, Parent, Teacher, Student, Board, Concept_map, Notes",Examination,"Change_of_plans, Task, Concentration, Confusion, Curiosity, Distraction, Errors, Set_a_goal, I commit, Importance, Intelligence, Store, Motivation, Plan, Problem, Plan, Program, Review, Postpone, Expiration, Challenge, Effort, Solution, Study, Success, Keep_track","Anxiety, Failure",52
2,3,"Biology, Equation, Physics, Informatics, Mathematics, Statistics, Science, Scientist, Innovation, Neuroscience","Teacher, Professor, University, Degree","Examination, Trial, Test, Average, Assessment","Creativity, Curiosity, Fun, Challenge, Model, Know, Development, Passion, Work","Anxiety, Attachment, Well being, Cognition, Emotion, Mind, Personality, Psychology, Psychopathology, Psychotherapy, Therapist, Behavior",41
3,4,"Biology, Equation, Physics, Mathematics, Numbers, Statistics, Technology, Graphic, Science, Discovery, Neuroscience","Teacher, Student, Professor, School, University, Degree","Examination, Exercise, Vote","Task, Knowledge, Creativity, Curiosity, Challenge, Future, Model, Relation, Work","Anxiety, Well being, Cognition, Emotion, Frustration, Mind, Boredom, Personality, Psychology, Therapy, Stress, Behavior",42
4,5,"Analyses, Equation, Physics, Informatics, Mathematics, Numbers, Statistics, Data, Hypothesis, Laboratory, Logic, Research, Science, Scientist, Scientist, Discovery, Experiment, Innovation","Companions, Teacher, Parents, School","Examination, Vote","Knowledge, Creativity, Problem, Programming, Challenge, Future, Rules","Anxiety, Art, Well being, Counselling, Failure, Embarrassment, Meditation, Panic, Fear, Stress, Life",43


In [6]:
latex = (
    df_summary.to_latex(
        multirow=True,
        position="!hbpt",
        caption="Cue words for different samples.",
        label="tab: cue_words_summary",
        column_format="c|p{2.8cm}|p{3.2cm}|p{2.3cm}|p{3.2cm}|p{2.8cm}",
        index=False,
    )
    .replace("_", " ")
    .replace("NaN", "")
)
latex = latex.replace(
    r"\begin{table}[!hbpt]", "\\begin{table}[!hbpt]\n\\scriptsize\n\\centering"
)

with open(FORMATTING / "cue_words_summary_latex.txt", "w") as f:
    f.write(latex)

## Flattened cue words and per-category counts

In [9]:
# Flattened, normalised list of English cue words per set (used to italicise hubs that are cue
# words in the feature tables below) plus the per-category cue-word counts. Both are derived from
# `df_summary`, so the cue words are built only once, here.
category_cols = list(CATEGORY_LISTS.keys())
WORD_FIXES = {"Examination": "Exam", "Well being": "Wellbeing"}

cue_words_df = pd.DataFrame({"Set": df_summary["Set"].astype(str)})
cue_words_df["words"] = df_summary[category_cols].apply(
    lambda row: sorted(
        {
            WORD_FIXES.get(w.strip(), w.strip())
            for cell in row.dropna()
            for w in str(cell).split(",")
            if w.strip()
        }
    ),
    axis=1,
)

cue_words_counts = df_summary[["Set"] + category_cols].copy()
for col in category_cols:
    cue_words_counts[col] = cue_words_counts[col].apply(count_words)
cue_words_counts["Total"] = df_summary["Total"]
cue_words_counts

,Set,STEM Disciplines,Academic Environment,Evaluation,Learning and Motivation,Mental Health,Total
0,1,27,10,5,3,5,51
1,2,15,7,1,26,2,52
2,3,10,4,5,9,12,41
3,4,11,6,3,9,12,42
4,5,18,4,2,7,11,43


# Network feature tables

## Cue-word ↔ sample correspondence

In [10]:
set_words_sample = {
    ("1"): ["Experts", "H-S South"],
    ("5"): ["H-Anx H-S", "L-Anx H-S", "Physics"],
    ("2", "3", "4", "5"): [
        "H-Anx Psy",
        "L-Anx Psy",
        "GPT H-Anx Psy",
        "GPT L-Anx Psy",
        "GPT Physics",
        "GPT H-Anx H-S",
        "GPT L-Anx H-S",
    ],
}
sample_to_sets = {
    sample: list(sets_words)
    for sets_words, samples in set_words_sample.items()
    for sample in samples
}

## Functions

In [11]:
def translate_and_italicize_hubs(
    df,
    col="Hubs",
    cue_words=None,
    source="it",
    target="en",
    set_words_sample=set_words_sample,
):

    if cue_words is None:
        cue_words = []

    new_df = df.copy()

    # Ensure lists are lists, not stringified lists
    new_df[col] = new_df[col].apply(ast.literal_eval)

    # Collect unique node words (before parentheses)
    unique_nodes = list(
        {
            re.search(r"^[^()]+", el).group(0).strip().capitalize()
            for lst in new_df[col]
            for el in lst
        }
    )

    # Join for one-shot translation
    joined = "\n".join(unique_nodes)

    # Translate in one API call
    translated_blob = GoogleTranslator(source=source, target=target).translate(joined)
    translated_list = translated_blob.split("\n")

    # Map original capitalized node → translated node
    translation_map = dict(zip(unique_nodes, translated_list))

    # Build final processed column
    processed_col = []
    n_hubs_records = []

    for idx, lst in enumerate(new_df[col]):
        processed_list = []

        for el in lst:
            node = re.search(r"^[^()]+", el).group(0).strip()
            degree = re.search(r"\(\d+\)", el).group(0)

            translated_node = translation_map[node.capitalize()]

            set_for_sample = sample_to_sets.get(new_df.loc[idx, "Sample"], [])
            cue_words_for_this_sample = (
                cue_words_df.loc[cue_words_df["Set"].isin(set_for_sample), "words"]
                .explode()
                .tolist()
            )

            if translated_node in cue_words_for_this_sample:
                formatted = f"\\textit{{{translated_node}}} {degree}"
            else:
                formatted = f"{translated_node} {degree}"

            processed_list.append(formatted)

            # Join as clean string
        processed_col.append(", ".join(processed_list))

        # Count hubs
        n_hubs_records.append({"Sample": new_df.loc[idx, "Sample"], "N Hubs": len(lst)})

    new_df[col] = processed_col
    n_hubs_df = pd.DataFrame(n_hubs_records)

    return new_df, n_hubs_df

In [12]:
def mapping_cols_df(features_df):
    keyword_map = {
        "ansia": "anxiety",
        "benessere": "wellbeing",
        "computer": "computer science",
        "esame": "exam",
        "fisica": "physics",
        "scuola": "school",
        "informatica": "computer science",
        "insegnante": "teacher",
        "matematica": "mathematics",
        "math": "mathematics",
        "professore": "professor",
        "scienza": "science",
        "scuola": "school",
        "scienziato": "scientist (male)",
        "scienziata": "scientist (female)",
        "statistica": "statistics",
        "studente": "student",
        "universita": "university",
        "ricerca": "research",
        "voto": "grade",
        "scientist": "scientist (m/f)",
    }
    sample_map = {
        "experts_2019": "Experts",
        "liceo_2019": "H-S South",
        "liceo_2025/high_anxiety": "H-Anx H-S",
        "liceo_2025/low_anxiety": "L-Anx H-S",
        "gpt_oss_high_school/high_anxiety": "GPT H-Anx H-S",
        "gpt_oss_high_school/low_anxiety": "GPT L-Anx H-S",
        "physics_students_2025": "Physics",
        "gpt_oss_physics": "GPT Physics",
        "psychology_students_unfiltered/high_anxiety": "H-Anx Psy",
        "psychology_students_unfiltered/low_anxiety": "L-Anx Psy",
        "gpt_oss_psychology/high_anxiety": "GPT H-Anx Psy",
        "gpt_oss_psychology/low_anxiety": "GPT L-Anx Psy",
    }

    if "keyword" in features_df.columns:
        features_df["keyword"] = (
            features_df["keyword"].map(keyword_map).fillna(features_df["keyword"])
        )

    features_df["Sample"] = (
        features_df["Sample"].map(sample_map).fillna(features_df["Sample"])
    )

    features_df_cleaned, n_hubs_df = translate_and_italicize_hubs(
        features_df, cue_words=cue_words_df
    )

    return features_df_cleaned, n_hubs_df

## Feature keyword fmns

In [13]:
all_keyword_fmns_features = pd.read_csv(
    f"{path_rassereno_analysis}/features_keyword_fmns.csv"
)

In [14]:
keyword_fmns_features_cleaned, hubs_df = mapping_cols_df(all_keyword_fmns_features)

In [15]:
keyword_fmns_features_cleaned = keyword_fmns_features_cleaned.rename(
    columns={"Hubs": "Hubs (top 5\\% of degree distribution)"}
)

In [16]:
keywords = [
    "mathematics",
    "statistics",
    "computer science",
    "physics",
    "science",
    "research",
    "exam",
    "grade",
    "professor",
    "teacher",
    "scientist",
    "scientist (male)",
    "scientist (female)",
    "school",
    "university",
    "anxiety",
    "wellbeing",
]

In [22]:
for keyword in keywords:
    df_keyword = (
        keyword_fmns_features_cleaned[
            keyword_fmns_features_cleaned["keyword"] == keyword
        ]
        .drop(columns=["keyword"])
        .to_latex(
            index=False,
            escape=False,
            float_format="%.2f",
            position="!hbpt",
            column_format="lccccp{9cm}",
            caption=f"Network features for the semantic frame of \\textit{{\\textbf{{{keyword.capitalize()}}}}}. For each hub, its degree is reported in parentheses. The italicised hubs are part of the cue words.",
            label=f"tab:network_features_{keyword}",
        )
    )
    df_latex = df_keyword.replace(
        r"\begin{table}[!hbpt]", f"\\begin{{table}}[!hbpt]\n\\footnotesize\n\\centering"
    )

    with open(f"{path_rassereno_analysis}/features_keyword_fmns_latex.txt", "w") as f:
        f.write(f"{df_latex}\n")

## Feature total fmns

In [18]:
all_tot_fmns_features = pd.read_csv(f"{path_rassereno_analysis}/features_tot_fmns.csv")

In [19]:
tot_fmns_features_cleaned, hubs = mapping_cols_df(all_tot_fmns_features)

In [20]:
tot_fmns_features_cleaned = tot_fmns_features_cleaned.rename(
    columns={"Hubs": "Hubs (top 1\\% of degree distribution)"}
)

In [21]:
tot_features_latex = tot_fmns_features_cleaned.to_latex(
    index=False,
    escape=False,
    float_format="%.2f",
    position="!hbpt",
    column_format="lccccp{9.5cm}",
    caption="Network features for the total semantic frame of each sample. For each hub, its degree is reported in parentheses. The italicised hubs are part of the cue words.",
    label="tab:network_features_total",
)
tot_features_latex = tot_features_latex.replace(
    r"\begin{table}[!hbpt]", f"\\begin{{table}}[!hbpt]\n\\footnotesize\n\\centering"
)

with open(f"{path_rassereno_analysis}/features_tot_fmns_latex.txt", "w") as f:
    f.write(f"{tot_features_latex}\n")

## Keywords

In [ ]:
keywords_rassereno_eng = [
    "mathematics",
    "statistics",
    "physics",
    "science",
    "computer",
    "research",
    "exam",
    "grade",
    "professor",
    "teacher",
    "scientist",
    "school",
    "university",
    "anxiety",
    "wellbeing",
]

cap_keywords = [w.capitalize().replace("'", "") for w in keywords_rassereno_eng]

dict_words = {
    "Experiment 1": [cap_keywords[:6]],
    "Experiment 2": [cap_keywords[6:13]],
    "Experiment 3": [cap_keywords[13:]],
}
df_words = pd.DataFrame.from_dict(dict_words)
df_words.to_latex()

"\\begin{tabular}{llll}\n\\toprule\n & Experiment 1 & Experiment 2 & Experiment 3 \\\\\n\\midrule\n0 & ['Mathematics', 'Statistics', 'Physics', 'Science', 'Computer', 'Research'] & ['Exam', 'Grade', 'Professor', 'Teacher', 'Scientist', 'School', 'University'] & ['Anxiety', 'Wellbeing'] \\\\\n\\bottomrule\n\\end{tabular}\n"